<h1> Data Preperation

Load in Labelled Windows

In [ ]:
from label_reading_tools import labelled_data_loading

width = 80
height = 120

root_folder = f"G:/CNN Formatted Data - {width} x {height}"

height += 1
width += 1

# windows not fitting the correct dimentions for a CNN are immediatly filtered in this loading process
# available labels = ["c", "p", "e", "n", "u", "f"]

win_raw, win_filtered, labels = labelled_data_loading(root_folder, width, height, ["c", "e", "n", "f"])

import numpy as np
num_classes = len(np.unique(labels))
print(f'Total Classes: {num_classes}')

from collections import Counter
print(f'Class Counts: {Counter(labels)}')

Spectrograms for the Windows

In [ ]:
from copy import deepcopy

stacks = []
for win in win_raw:
    stack = None
    for i in range(len(win[0])):
        if stack is None:
            stack = deepcopy(win[:, i])
        else:
            stack = stack + win[:, i]

    stack = stack / len(win[0])
    stacks.append(stack)

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt

spectrograms = []
for stack in stacks:
    f, t, Sxx = signal.spectrogram(stack, 1000, nperseg=12, mode= "magnitude")
    spectrograms.append(Sxx)

    # fig = plt.figure()
    # ax = fig.add_subplot()
    # ax.set_ylim(0, 500)
    # cmap = plt.colormaps['viridis']
    # img1 = plt.pcolormesh(t, f, Sxx, cmap=cmap)
    #
    # plt.ylabel('Frequency [Hz]')
    # plt.xlabel('Time [sec]')
    # fig.colorbar(img1, label= "Nano Strain per Second [nm/m/s]")
    # break

Feature Extraction

In [ ]:
from numpy.fft import rfftfreq
from scipy.fft import rfft
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler, MinMaxScaler

features = []
for win, filt_win, stack, spectrogram in zip(win_raw, win_filtered, stacks, spectrograms):

    # window features
    win_amp_max = np.max(win)
    win_amp_min = np.min(win)

    win_filt_amp_max = np.max(filt_win)
    win_filt_amp_min = np.min(filt_win)

    #stacked channel features

    stack_mean = np.mean(stack)
    stack_std = np.std(stack)
    stack_skew = skew(stack)
    stack_kurtosis = kurtosis(stack)

    # stacked channel spectra

    fs = 1000

    N = len(stack)
    yf = rfft(stack)
    xf = rfftfreq(N, 1/fs)

    spectra_kurtosis = kurtosis(np.abs(yf))
    spectra_skew = skew(np.abs(yf))
    spectra_domFreq = xf[np.where(np.abs(yf) == np.max(np.abs(yf)))[0]][0]
    spectra_domFreq_amp = np.max(np.abs(yf))

    feature = [win_amp_max, win_amp_min, win_filt_amp_max, win_filt_amp_min, stack_mean, stack_std, stack_skew, stack_kurtosis, spectra_skew, spectra_kurtosis, spectra_domFreq, spectra_domFreq_amp]
    features.append(feature)

In [ ]:
from numpy.fft import rfftfreq
from scipy.fft import rfft
from scipy.stats import skew, kurtosis
import numpy as np

features = []

fs = 1000

for win, filt_win, stack, spectrogram in zip(
        win_raw,
        win_filtered,
        stacks,
        spectrograms):

    # -------------------------
    # Window features
    # -------------------------

    win_amp_max = np.max(win)
    win_amp_min = np.min(win)

    win_filt_amp_max = np.max(filt_win)
    win_filt_amp_min = np.min(filt_win)

    win_ptp = np.ptp(win)

    # -------------------------
    # Stack statistics
    # -------------------------

    stack_mean = np.mean(stack)
    stack_std = np.std(stack)

    stack_skew = skew(stack)
    stack_kurtosis = kurtosis(stack)

    # RMS
    stack_rms = np.sqrt(np.mean(stack**2))

    # Energy
    stack_energy = np.sum(stack**2)

    # Log Energy
    stack_log_energy = np.log10(stack_energy + 1e-10)

    # Crest Factor
    stack_crest_factor = (
            np.max(np.abs(stack))
            / (stack_rms + 1e-10)
    )

    # Zero Crossing Rate
    stack_zcr = np.sum(
        stack[:-1] * stack[1:] < 0
    )

    # -------------------------
    # FFT
    # -------------------------

    N = len(stack)

    yf = rfft(stack)
    xf = rfftfreq(N, 1/fs)

    mag = np.abs(yf)

    # Existing features

    spectra_skew = skew(mag)
    spectra_kurtosis = kurtosis(mag)

    idx_dom = np.argmax(mag)

    spectra_domFreq = xf[idx_dom]
    spectra_domFreq_amp = mag[idx_dom]

    # -------------------------
    # Spectral Entropy
    # -------------------------

    p = mag / (np.sum(mag) + 1e-10)

    spectral_entropy = -np.sum(
        p * np.log2(p + 1e-10)
    )

    # -------------------------
    # Spectral Centroid
    # -------------------------

    spectral_centroid = (
            np.sum(xf * mag)
            / (np.sum(mag) + 1e-10)
    )

    # -------------------------
    # Spectral Bandwidth
    # -------------------------

    spectral_bandwidth = np.sqrt(
        np.sum(
            ((xf - spectral_centroid) ** 2) * mag
        )
        / (np.sum(mag) + 1e-10)
    )

    # -------------------------
    # Frequency Percentiles
    # -------------------------

    cumulative = np.cumsum(mag)
    cumulative /= cumulative[-1]

    f25 = xf[np.searchsorted(cumulative, 0.25)]
    f50 = xf[np.searchsorted(cumulative, 0.50)]
    f75 = xf[np.searchsorted(cumulative, 0.75)]
    f90 = xf[np.searchsorted(cumulative, 0.90)]

    # -------------------------
    # Energy above frequency thresholds
    # -------------------------

    spectral_energy = np.sum(mag**2)

    # > 10 Hz
    mask_10 = xf > 10

    energy_gt_10Hz = np.sum(mag[mask_10]**2)

    energy_ratio_gt_10Hz = (
            energy_gt_10Hz /
            (spectral_energy + 1e-10)
    )

    # > 50 Hz
    mask_50 = xf > 50

    energy_gt_50Hz = np.sum(mag[mask_50]**2)

    energy_ratio_gt_50Hz = (
            energy_gt_50Hz /
            (spectral_energy + 1e-10)
    )

    # > 100 Hz
    mask_100 = xf > 100

    energy_gt_100Hz = np.sum(mag[mask_100]**2)

    energy_ratio_gt_100Hz = (
            energy_gt_100Hz /
            (spectral_energy + 1e-10)
    )

    # -------------------------
    # Feature Vector
    # -------------------------

    feature = [

        # Window
        win_amp_max,
        win_amp_min,
        win_filt_amp_max,
        win_filt_amp_min,
        win_ptp,

        # Stack statistics
        stack_mean,
        stack_std,
        stack_skew,
        stack_kurtosis,
        stack_rms,
        stack_energy,
        stack_log_energy,
        stack_crest_factor,
        stack_zcr,

        # Spectrum
        spectra_skew,
        spectra_kurtosis,
        spectra_domFreq,
        spectra_domFreq_amp,
        spectral_entropy,
        spectral_centroid,
        spectral_bandwidth,

        # Spectral percentiles
        f25,
        f50,
        f75,
        f90,

        # Energy above thresholds
        energy_gt_10Hz,
        energy_ratio_gt_10Hz,

        energy_gt_50Hz,
        energy_ratio_gt_50Hz,

        energy_gt_100Hz,
        energy_ratio_gt_100Hz
    ]

    features.append(feature)

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
features_scaled = scaler.fit_transform(features)

In [ ]:
feature_names = [

    # Window features
    "win_amp_max",
    "win_amp_min",
    "win_filt_amp_max",
    "win_filt_amp_min",
    "win_ptp",

    # Stack statistics
    "stack_mean",
    "stack_std",
    "stack_skew",
    "stack_kurtosis",
    "stack_rms",
    "stack_energy",
    "stack_log_energy",
    "stack_crest_factor",
    "stack_zcr",

    # Spectral statistics
    "spectra_skew",
    "spectra_kurtosis",
    "spectra_domFreq",
    "spectra_domFreq_amp",
    "spectral_entropy",
    "spectral_centroid",
    "spectral_bandwidth",

    # Spectral percentiles
    "f25",
    "f50",
    "f75",
    "f90",

    # Energy above thresholds
    "energy_gt_10Hz",
    "energy_ratio_gt_10Hz",

    "energy_gt_50Hz",
    "energy_ratio_gt_50Hz",

    "energy_gt_100Hz",
    "energy_ratio_gt_100Hz"

]

In [ ]:
import pandas as pd
import seaborn as sns

df = pd.DataFrame(features, columns=feature_names)

for feature in df.columns:

    plt.figure(figsize=(8,4))

    sns.violinplot(
        x=labels,
        y=df[feature]
    )

    plt.title(feature)
    plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)

rf.fit(df.values, labels)

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df)

pca = PCA()
pca.fit(X)

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(len(feature_names))],
    index=feature_names
)

print(loadings["PC1"].abs().sort_values(ascending=False))

Normalising Windows

In [116]:
# this normalises windows (between -1, 1) individually as normalising as a group leads to inconsistent normalization
import numpy as np

win_raw_normalised = []
win_filtered_normalised = []
spectrograms_normalised = []
stacks_normalised = []
for w, f, s, st in zip(win_raw, win_filtered, spectrograms, stacks):

    minVal = w.min()
    maxVal = w.max()

    w_normalised = (((w - minVal) / (maxVal - minVal)) - 0.5) * 2
    win_raw_normalised.append(w_normalised)

    minVal = f.min()
    maxVal = f.max()

    f_normalised = (((f - minVal) / (maxVal - minVal)) - 0.5) * 2
    win_filtered_normalised.append(f_normalised)

    minVal = s.min()
    maxVal = s.max()

    s_normalised = (((s - minVal) / (maxVal - minVal)) - 0.5) * 2
    spectrograms_normalised.append(s_normalised)

    minVal = st.min()
    maxVal = st.max()

    st_normalised = (((st - minVal) / (maxVal - minVal)) - 0.5) * 2
    stacks_normalised.append(st_normalised)

win_raw_normalised = np.array(win_raw_normalised)
win_filtered_normalised = np.array(win_filtered_normalised)
spectrograms_normalised = np.array(spectrograms_normalised)
stacks_normalised = np.array(stacks_normalised)

Class Selection and One Hot Encoding

In [ ]:
# footsteps vs other
for l in labels:
    if l != "f":
        labels[labels == l] = "o"

In [ ]:
# potential cracking vs other
for l in labels:
    if l != "c":
        labels[labels == l] = "o"

In [117]:
from keras.src.utils import to_categorical
labels = np.array(labels)

unique = np.unique(labels)
print(unique)

for i, u in enumerate(unique):
    labels[labels == u] = i

labels_ohe = to_categorical(labels, num_classes=num_classes)

print(labels_ohe[0])

['c' 'e' 'f' 'n']
[1. 0. 0. 0.]


Balancing Classes (Experiment with Binary vs Multiclass)

In [118]:
# general purpose balancing
win_raw_normalised_balanced = []
win_filtered_normalised_balanced = []
spectrograms_normalised_balanced = []
labels_ohe_balanced = []
labels_balanced = []

smallest_class = min(Counter(labels).values())

for i in range(num_classes):
    count = 0
    for w, wf, s, ohe, l in zip(win_raw_normalised, win_filtered_normalised, spectrograms_normalised, labels_ohe, labels):
        if count == smallest_class:
            break
        else:
            if l == f'{i}':
                labels_balanced.append(l)
                labels_ohe_balanced.append(ohe)
                spectrograms_normalised_balanced.append(s)
                win_raw_normalised_balanced.append(w)
                win_filtered_normalised_balanced.append(wf)
                count+=1

In [121]:
Counter(labels_balanced)

Counter({'0': 921, '1': 921, '2': 921, '3': 921})

Reshape Window For CNN

In [119]:
win_raw_normalised = np.array(win_raw_normalised).astype(np.float32)
win_raw_normalised = win_raw_normalised.reshape(-1, 1, height, width)

win_filtered_normalised = np.array(win_filtered_normalised).astype(np.float32)
win_filtered_normalised = win_filtered_normalised.reshape(-1, 1, height, width)

spectrograms_normalised = np.array(spectrograms_normalised).astype(np.float32)
spectrograms_normalised = spectrograms_normalised.reshape(-1, 1, height, width)

ValueError: cannot reshape array of size 770210 into shape (1,121,81)

In [120]:
win_raw_normalised_balanced = np.array(win_raw_normalised_balanced).astype(np.float32)
win_raw_normalised_balanced = win_raw_normalised_balanced.reshape(-1, 1, height, width)

win_filtered_normalised_balanced = np.array(win_filtered_normalised_balanced).astype(np.float32)
win_filtered_normalised_balanced = win_filtered_normalised_balanced.reshape(-1, 1, height, width)

spectrograms_normalised_balanced = np.array(spectrograms_normalised_balanced).astype(np.float32)
spectrograms_normalised_balanced = spectrograms_normalised_balanced.reshape(-1, 1, 7, 10)

<h1> Machine Learning

In [143]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# random seed
random_seed = 2

# train validation split
win_raw_train, temp_win, labels_train_enc, temp_label = train_test_split(np.array(win_raw_normalised_balanced), np.array(labels_ohe_balanced),
                                                                         test_size=0.3)
win_raw_val, win_raw_test, labels_val_enc, labels_test_enc = train_test_split(temp_win, temp_label,
                                                                              test_size=0.5)

print(f"Train: {len(win_raw_train)}")
print(f"Validation: {len(win_raw_val)}")
print(f"Test: {len(win_raw_test)}")

train_dataset = TensorDataset(torch.from_numpy(win_raw_train), torch.from_numpy(labels_train_enc))
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=16)

val_dataset = TensorDataset(torch.from_numpy(win_raw_val), torch.from_numpy(labels_val_enc))
val_loader = DataLoader(val_dataset, shuffle=True, batch_size=16)

test_dataset = TensorDataset(torch.from_numpy(win_raw_test), torch.from_numpy(labels_test_enc))
test_loader = DataLoader(test_dataset, shuffle=True, batch_size=16)

Train: 2578
Validation: 553
Test: 553


In [137]:
import torch.nn as nn
import torch.nn.functional as F

class MyFirstNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 12, 5) # (input (121) - kernal (5) / stride (1)) + 1 -> (12, 117, 77)
        self.pool = nn.MaxPool2d(2, 2) # (12, 58, 38) pooling always floors
        self.conv2 = nn.Conv2d(12, 24, 5) #(24, 54, 34) -> pool again (24, 27, 17)
        self.fc1 = nn.Linear(24 * 27 * 17, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 4) # input and final output are the only fixed values

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [135]:
class AdvancedNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        # (input (121) - kernal (3) / stride (1)) + 1 -> (32, 119, 79)
        # (32, 59, 39) pooling always floors

        self.conv1 = nn.Conv2d(1, 32, 3) #119, 79
        self.conv2 = nn.Conv2d(32, 64, 3) #117, 77
        self.bthNorm = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2) #58, 38

        self.conv3 = nn.Conv2d(64, 128, 3) #56, 36
        self.conv4 = nn.Conv2d(128, 256, 3) #54, 34
        self.conv5 = nn.Conv2d(256, 512, 3) #52, 32
        self.bthNorm2 = nn.BatchNorm2d(512)

        self.fc1 = nn.Linear((512 * 26 * 16), 1024)
        self.drp = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(1024, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 5)



    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.bthNorm(x)
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.bthNorm2(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = self.drp(F.relu(self.fc1(x)))
        x = self.drp(F.relu(self.fc2(x)))
        x = self.drp(F.relu(self.fc3(x)))
        x = F.softmax(self.fc4(x))
        return x

Network Training

In [144]:
from PyTorch_Darknet53_master.model import darknet53

import torch.nn as nn
import torch.optim as optim

model_save = './models/testing.pth'

num_epochs = 50
# net = darknet53(num_classes)
net = MyFirstNeuralNet()

if torch.cuda.is_available():
    net = net.cuda()

loss_function = nn.CrossEntropyLoss()

# optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
# optimizer = optim.Adam(net.parameters(), lr=0.001)
optimizer = optim.Adam(net.parameters(), lr=0.0001)

min_valid_loss = np.inf

for e in range(num_epochs):
    train_loss = 0.0
    net.train()  # Optional when not using Model Specific layer
    for data, train_labels in train_loader:
        if torch.cuda.is_available():
            data, train_labels = data.cuda(), train_labels.cuda()

        optimizer.zero_grad()

        target = net(data)
        loss = loss_function(target, train_labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    valid_loss = 0.0
    net.eval()  # Optional when not using Model Specific layer
    with torch.no_grad():
        for data, val_labels in val_loader:
            if torch.cuda.is_available():
                data, val_labels = data.cuda(), val_labels.cuda()

            target = net(data)
            loss = loss_function(target, val_labels)
            valid_loss = loss.item() * data.size(0)

    print(f'Epoch {e + 1} \t\t Training Loss: {train_loss / len(train_loader)} \t\t Validation Loss: {valid_loss / len(val_loader)}')

    if min_valid_loss > valid_loss:
        print(f'Validation Loss Decreased({min_valid_loss:.6f}--->{valid_loss:.6f}) \t Saving The Model')
        min_valid_loss = valid_loss
        # Saving State Dict
        torch.save(net.state_dict(), model_save)

Epoch 1 		 Training Loss: 1.069226164103451 		 Validation Loss: 0.29584947271006445
Validation Loss Decreased(inf--->10.354732) 	 Saving The Model
Epoch 2 		 Training Loss: 0.7484288728292198 		 Validation Loss: 0.12850236046527114
Validation Loss Decreased(10.354732--->4.497583) 	 Saving The Model
Epoch 3 		 Training Loss: 0.6846935085711188 		 Validation Loss: 0.30988622696271967
Epoch 4 		 Training Loss: 0.6473914900779175 		 Validation Loss: 0.1969745574252946
Epoch 5 		 Training Loss: 0.6185864240907881 		 Validation Loss: 0.19105830016945088
Epoch 6 		 Training Loss: 0.5928122749238905 		 Validation Loss: 0.2985924309651766
Epoch 7 		 Training Loss: 0.5812872228338232 		 Validation Loss: 0.17149198765733412
Epoch 8 		 Training Loss: 0.5660678035117509 		 Validation Loss: 0.22880366487162454
Epoch 9 		 Training Loss: 0.5481134924520019 		 Validation Loss: 0.1694744789706809
Epoch 10 		 Training Loss: 0.5333765504491466 		 Validation Loss: 0.17480469439677626
Epoch 11 		 Training L

Network Testing

In [145]:
import numpy as np
import torch

# train_net = darknet53(num_classes)
train_net = MyFirstNeuralNet()

train_net.load_state_dict(torch.load(model_save))

correct = 0
total = 0

# Confusion matrix: rows = actual, columns = predicted
confusion = np.zeros((num_classes, num_classes), dtype=int)

train_net.eval()

with torch.no_grad():
    for data in test_loader:
        images, test_labels = data

        outputs = train_net(images)
        _, predicted = torch.max(outputs.data, 1)

        total += test_labels.size(0)

        for l, p in zip(test_labels, predicted):
            actual = torch.argmax(l).item()  # convert one-hot label to class index
            pred = p.item()

            confusion[actual, pred] += 1

            if actual == pred:
                correct += 1

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

print("\nConfusion Matrix")
print("(rows=actual, cols=predicted)")
print(confusion)

# Per-class statistics
for c in range(num_classes):
    true_positive = confusion[c, c]
    false_positive = confusion[:, c].sum() - true_positive
    false_negative = confusion[c, :].sum() - true_positive

    print(
        f"Class {c}: "
        f"TP={true_positive}, "
        f"FP={false_positive}, "
        f"FN={false_negative}"
    )

Test Accuracy: 77.22%

Confusion Matrix
(rows=actual, cols=predicted)
[[144   3   5   5]
 [  1  95  10  20]
 [  0  17  70  35]
 [  1  15  14 118]]
Class 0: TP=144, FP=2, FN=13
Class 1: TP=95, FP=35, FN=31
Class 2: TP=70, FP=29, FN=52
Class 3: TP=118, FP=60, FN=30


In [146]:
for c in range(num_classes):

    true_positive = confusion[c, c]

    total_actual = confusion[c, :].sum()

    class_accuracy = (
        true_positive / total_actual
        if total_actual > 0 else 0
    )

    print(
        f"Class {c}: "
        f"Accuracy = {100*class_accuracy:.2f}% "
        f"({true_positive}/{total_actual})"
    )

Class 0: Accuracy = 91.72% (144/157)
Class 1: Accuracy = 75.40% (95/126)
Class 2: Accuracy = 57.38% (70/122)
Class 3: Accuracy = 79.73% (118/148)


In [147]:
for c in range(num_classes):

    tp = confusion[c, c]

    fp = confusion[:, c].sum() - tp

    fn = confusion[c, :].sum() - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    print(
        f"Class {c}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )


Class 0: Precision=0.986, Recall=0.917, F1=0.950
Class 1: Precision=0.731, Recall=0.754, F1=0.742
Class 2: Precision=0.707, Recall=0.574, F1=0.633
Class 3: Precision=0.663, Recall=0.797, F1=0.724
